# Nemotron Reasoning Challenge — Colab Pro (A100)

Train a LoRA adapter on **Nemotron-3-Nano-30B** using Colab Pro A100, then download `submission.zip` for Kaggle.

**Before running:** Set runtime to **A100 GPU** (Runtime > Change runtime type > A100 GPU).

| Phase | Script | What it does |
|-------|--------|--------------|
| 1 | `scripts/01_eda.py` | Classify prompts, token-length stats, EDA report |
| 2 | `scripts/02_prepare_data.py` | Synthetic data + optional API CoT → `train_sft.jsonl` |
| 3 | `scripts/03_train_lora.py` | bf16 LoRA SFT (rank 32) on Nemotron-3-Nano-30B |
| 4 | `scripts/05_package_submission.py` | Zip LoRA adapter → `submission.zip` |
| 5 | Download | `submission.zip` to local machine, submit to Kaggle |

## Setup: upload project files + competition data

Upload `udacity_upload.zip` and `train.csv` to Colab, or mount Google Drive.

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path
from google.colab import files, drive

WORK_ROOT = Path("/content/project").resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Option A: upload zip directly (recommended)
# ---------------------------------------------------------------------------
print("Upload udacity_upload.zip...")
uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall(WORK_ROOT)
        print(f"Extracted {name} to {WORK_ROOT}")

# ---------------------------------------------------------------------------
# Upload train.csv
# ---------------------------------------------------------------------------
data_dir = WORK_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "reports").mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "synthetic").mkdir(parents=True, exist_ok=True)

if not (data_dir / "train.csv").is_file():
    print("Upload train.csv...")
    uploaded2 = files.upload()
    for name in uploaded2:
        Path(name).rename(data_dir / name)

os.chdir(WORK_ROOT)
sys.path.insert(0, str(WORK_ROOT))

assert (WORK_ROOT / "scripts" / "01_eda.py").is_file(), "scripts/ not found"
assert (data_dir / "train.csv").is_file(), "train.csv not found"
print("cwd:", os.getcwd())
print("train.csv:", (data_dir / "train.csv").stat().st_size, "bytes")

## Configuration

In [ ]:
import torch

MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # or paste token string here

# A100 (40 GB VRAM): 4-bit QLoRA (30B bf16 = 60 GB, doesn't fit in 40 GB without quant)
GPU_PROFILE = "a100"
USE_BF16_FULL = False
TRAIN_MAX_SEQ = 4096
TRAIN_MAX_MEMORY_JSON = None

LORA_TARGET_MODE = "kaggle_nemotron"
LORA_ALPHA = 32
TRAIN_BATCH = 4
GRAD_ACCUM = 4
NUM_EPOCHS = 2.0
LR = 1e-4

SKIP_COT = False
SYNTHETIC_PER_KIND = 400

print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
print(f"GPU_PROFILE: {GPU_PROFILE} | bf16: {USE_BF16_FULL} | max_seq: {TRAIN_MAX_SEQ}")
print(f"Model: {MODEL_ID}")

## Install dependencies

In [ ]:
import subprocess, sys, os, re

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def pip_try(*args):
    try:
        pip_install(*args)
        return True
    except subprocess.CalledProcessError:
        return False

pip_install("-U", "pip", "setuptools", "wheel")
pip_install(
    "transformers>=4.45,<5",
    "peft>=0.12",
    "trl>=0.12",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "psutil",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm",
    "huggingface_hub",
    "ninja",
)

# mamba / causal-conv1d: try GitHub wheels for the detected torch version
import torch
_torch_full = torch.__version__
_torch_mm = re.match(r"(\d+\.\d+)", _torch_full).group(1)
_torch_minor = int(_torch_mm.split(".")[1])
_cu = "cu12" if "cu12" in _torch_full else "cu11"
_py = f"cp{sys.version_info.major}{sys.version_info.minor}"
_abi_order = ["cxx11abiTRUE", "cxx11abiFALSE"] if _torch_minor >= 7 else ["cxx11abiFALSE", "cxx11abiTRUE"]
print(f"torch {_torch_full} | wheel tags: torch{_torch_mm}, {_cu}, {_abi_order[0]}, {_py}")

_GH_C = "https://github.com/Dao-AILab/causal-conv1d/releases/download"
_GH_M = "https://github.com/state-spaces/mamba/releases/download"

for pkg, versions in [
    ("causal-conv1d", [("v1.6.1", "causal_conv1d", "1.6.1"), ("v1.5.4", "causal_conv1d", "1.5.4")]),
    ("mamba-ssm", [("v2.3.1", "mamba_ssm", "2.3.1"), ("v2.2.4", "mamba_ssm", "2.2.4")]),
]:
    gh = _GH_C if "causal" in pkg else _GH_M
    installed = False
    for tag, wn, wv in versions:
        if installed:
            break
        for abi in _abi_order:
            url = f"{gh}/{tag}/{wn}-{wv}+{_cu}torch{_torch_mm}{abi}-{_py}-{_py}-linux_x86_64.whl"
            if pip_try(url):
                print(f"OK: {pkg} {wv} ({abi})")
                installed = True
                break
    if not installed:
        print(f"Building {pkg} from source...")
        os.environ["CAUSAL_CONV1D_FORCE_BUILD"] = "TRUE"
        os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
        pip_install("--no-build-isolation", "--no-deps", pkg)

pip_try("--prefer-binary", "nvidia-cutlass>=3.6", "nvidia-cutlass-dsl>=4.4")

print("\n--- Versions ---")
for p in ("torch", "transformers", "peft", "trl", "bitsandbytes", "mamba_ssm"):
    try: print(f"  {p}: {__import__(p).__version__}")
    except: print(f"  {p}: not found")

## Download base model

In [ ]:
from huggingface_hub import login, snapshot_download

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF login: OK")
else:
    try:
        login(add_to_git_credential=False)
        print("HF login: OK (interactive)")
    except Exception as e:
        print("HF login skipped:", e)

MODEL_PATH_LOCAL = snapshot_download(MODEL_ID, resume_download=True)
print("Model cached at:", MODEL_PATH_LOCAL)

## Auto-patch: fix Nemotron 4-bit compatibility

Patches the model file and training script to fix:
1. Mamba modules excluded from 4-bit quantization (Triton F.linear shape mismatch)
2. MoE index_add_ dtype mismatch (BFloat16 vs Float)
3. Expert dummy forward dtype (Byte from quantized params)

In [ ]:
import glob, os, shutil

# ---- Patch 1: fix modeling_nemotron_h.py (MoE dtype + expert dummy dtype) ----
model_files = glob.glob("/root/.cache/huggingface/hub/**/modeling_nemotron_h.py", recursive=True)
model_files += glob.glob("/root/.cache/huggingface/modules/**/modeling_nemotron_h.py", recursive=True)

for mf in set(model_files):
    with open(mf) as f:
        lines = f.readlines()
    changed = False
    for i, line in enumerate(lines):
        if "final_hidden_states.index_add_(0, token_indices, weighted_output)" in line and "weighted_output.to(" not in line:
            lines[i] = line.replace("weighted_output)", "weighted_output.to(final_hidden_states.dtype))")
            changed = True
        if ".to(expert_dtype)" in line:
            lines[i] = line.replace(".to(expert_dtype)", ".to(torch.bfloat16)")
            changed = True
    if changed:
        with open(mf, "w") as f:
            f.writelines(lines)
        pc = os.path.dirname(mf) + "/__pycache__"
        if os.path.exists(pc):
            shutil.rmtree(pc)
        print(f"PATCHED: {mf}")
    else:
        print(f"Already patched: {mf}")

# ---- Patch 2: fix 03_train_lora.py (skip Mamba modules from quantization) ----
script = str(WORK_ROOT / "scripts" / "03_train_lora.py")
with open(script) as f:
    code = f.read()

old_bnb = """    if quantize:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        base_kw["quantization_config"] = bnb"""

new_bnb = """    if quantize:
        _skip = ["lm_head"]
        if "nemotron" in model_path.lower():
            _skip += ["in_proj", "out_proj", "conv1d", "x_proj", "dt_proj"]
            print("[03_train_lora] Skipping quantization for Mamba modules", flush=True)
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=_skip,
        )
        base_kw["quantization_config"] = bnb"""

if old_bnb in code:
    code = code.replace(old_bnb, new_bnb)
    with open(script, "w") as f:
        f.write(code)
    print("PATCHED: 03_train_lora.py (Mamba quant skip)")
elif "llm_int8_skip_modules" in code:
    print("Already patched: 03_train_lora.py")
else:
    print("WARNING: could not patch 03_train_lora.py -- check manually")

print("\nAll patches applied.")

## Phase 1 — EDA

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "scripts/01_eda.py",
    "--data-dir", "data",
    "--report-dir", "data/reports",
    "--tokenizer-model", str(MODEL_PATH_LOCAL),
], check=True)

## Phase 1b — Solver pseudo-labels on test.csv

Run programmatic solvers over every prompt in the public `test.csv`. For
prompts the solver can crack (verified against ALL example pairs in the
prompt), we get free, distribution-perfect training examples. These get
folded into `train_sft.jsonl` in Phase 2.

This is the Kaggle Grandmasters' pseudo-labeling tip applied to a
generative-reasoning task with a deterministic teacher (the solver only
emits answers when it verifies them, so false positives are rare).

In [ ]:
import os, subprocess, sys

if os.path.isfile("data/test.csv"):
    subprocess.run([
        sys.executable, "scripts/02b_pseudolabel_test.py",
        "--test-csv", "data/test.csv",
        "--output", "data/pseudo_test.jsonl",
        "--report-dir", "data/reports",
    ], check=True)
    print(f"Pseudo-label file: data/pseudo_test.jsonl "
          f"(size: {os.path.getsize('data/pseudo_test.jsonl')} bytes)")
else:
    print("No data/test.csv found - skipping pseudo-label pass.")
    print("(This is OK if test.csv hasn't been uploaded yet.)")


## Phase 2 — Prepare SFT data

In [ ]:
import os, subprocess, sys

COT_BACKEND = "openai"
COT_MODEL = "gpt-4o"

prep_env = os.environ.copy()

if not SKIP_COT:
    api_key_var = "OPENAI_API_KEY" if COT_BACKEND == "openai" else "ANTHROPIC_API_KEY"
    if not prep_env.get(api_key_var):
        raise RuntimeError(
            f"SKIP_COT=False but {api_key_var} is not set. "
            f"Set it via os.environ['{api_key_var}'] = 'sk-...' before running this cell, "
            f"or set SKIP_COT=True to skip the API teacher path."
        )
    print(f"Using {COT_BACKEND}/{COT_MODEL} as API teacher (key tail: ...{prep_env[api_key_var][-4:]})")

cot_args = ["--skip-cot"] if SKIP_COT else [
    "--cot-backend", COT_BACKEND,
    "--cot-model", COT_MODEL,
    "--cot-max-tokens", "4096",
]

# If we ran 02b_pseudolabel_test.py earlier, fold those test-set
# pseudo-labels into the training mix for free domain alignment.
pseudo_label_args = []
pseudo_label_path = "data/pseudo_test.jsonl"
if os.path.isfile(pseudo_label_path):
    pseudo_label_args = ["--pseudo-label-file", pseudo_label_path]
    print(f"Including pseudo-labels from {pseudo_label_path}")

cmd = [
    sys.executable, "scripts/02_prepare_data.py",
    "--data-dir", "data",
    "--synthetic-dir", "data/synthetic",
    "--output", "data/train_sft.jsonl",
    "--tokenizer-model", str(MODEL_PATH_LOCAL),
    "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
] + cot_args + pseudo_label_args
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=prep_env)

## Phase 3 — LoRA training (A100 bf16)

In [ ]:
import gc, os, subprocess, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

train_env = os.environ.copy()
train_env["TOKENIZERS_PARALLELISM"] = "false"
train_env["NEMOTRON_KAGGLE_PATCHES"] = "0"
train_env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable, "scripts/03_train_lora.py",
    "--data-path", "data/train_sft.jsonl",
    "--output-dir", "lora_adapter",
    "--checkpoint-dir", "lora_output",
    "--model-path", str(MODEL_PATH_LOCAL),
    "--lora-target-mode", LORA_TARGET_MODE,
    "--lora-alpha", str(LORA_ALPHA),
    "--batch-size", str(TRAIN_BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--epochs", str(NUM_EPOCHS),
    "--lr", str(LR),
    "--max-seq-length", str(TRAIN_MAX_SEQ),
    "--force-peft",
    "--no-nemotron-kaggle-patches",
    "--dataloader-workers", "0",
]

gc.collect()
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(
    cmd, env=train_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

## GPU monitor (run in a separate cell while training)

Open a second Colab tab or right-click this cell to run it independently.

In [ ]:
import time, subprocess
for _ in range(60):
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    print(r.stdout.strip(), flush=True)
    time.sleep(30)

## Phase 4 — Package submission

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "scripts/05_package_submission.py",
    "--adapter-dir", "lora_adapter",
    "--output", "submission.zip",
], check=True)

zp = WORK_ROOT / "submission.zip"
print("submission.zip:", zp.is_file(), zp.stat().st_size if zp.is_file() else 0, "bytes")

## Phase 4b — (Optional) Upload adapter to Kaggle as a versioned Dataset

The competition accepts the adapter zip directly (Phase 5 below), but
publishing the adapter as a Kaggle Dataset gives you:

- Version history — every retraining run becomes a new dataset version,
  diffable on Kaggle's web UI.
- Easy attachment from any Kaggle inference notebook via "Add Data".
- Off-Colab backup (Colab session storage is volatile).

**Before running:** set `KAGGLE_DATASET_ID` to your slug. The first time
you publish, leave `FIRST_TIME = True`. On subsequent retrains, set it
to `False` and bump `VERSION_NOTES`.

Skip this entire cell if you only need the direct competition submit.

In [ ]:
# Set RUN_DATASET_UPLOAD = True to publish/version the adapter on Kaggle.
RUN_DATASET_UPLOAD = False

# Edit these to match your Kaggle username + desired dataset slug.
KAGGLE_DATASET_ID = "sebmontreal/nemotron-lora-adapter"
KAGGLE_DATASET_TITLE = "Nemotron LoRA Adapter (Reasoning Challenge)"

# First publish? Toggle True the first time only; flip False afterward.
FIRST_TIME = True
VERSION_NOTES = "v1 - initial upload"

if RUN_DATASET_UPLOAD:
    import os, subprocess, sys
    from pathlib import Path
    from google.colab import userdata

    # Pull KAGGLE_USERNAME / KAGGLE_KEY from Colab Secrets (Settings -> Secrets).
    # Falls back to ~/.kaggle/kaggle.json if you uploaded it manually.
    try:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        print("Loaded Kaggle credentials from Colab Secrets.")
    except Exception as e:
        if not (Path.home() / ".kaggle" / "kaggle.json").is_file():
            raise RuntimeError(
                "No Kaggle credentials. Either set KAGGLE_USERNAME and KAGGLE_KEY "
                "in Colab Secrets, or upload kaggle.json to ~/.kaggle/kaggle.json"
            ) from e
        print("Using ~/.kaggle/kaggle.json (Colab Secrets unavailable).")

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

    cmd = [
        sys.executable, "scripts/08_upload_adapter_dataset.py",
        "--adapter-dir", "lora_adapter",
        "--dataset-id", KAGGLE_DATASET_ID,
        "--title", KAGGLE_DATASET_TITLE,
    ]
    if FIRST_TIME:
        cmd.append("--first-time")
    else:
        cmd += ["--version-notes", VERSION_NOTES]

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("RUN_DATASET_UPLOAD is False - skipping. "
          "Set RUN_DATASET_UPLOAD=True after editing KAGGLE_DATASET_ID.")


## Phase 5 — Download submission + submit to Kaggle

Download `submission.zip` to your local machine, then submit via Kaggle CLI:

```bash
kaggle competitions submit -c nvidia-nemotron-model-reasoning-challenge -f submission.zip -m "Colab A100 trained adapter"
```

In [ ]:
from google.colab import files

files.download(str(WORK_ROOT / "submission.zip"))

## (Optional) Phase 6 — K-fold cross-validation

> **Skip unless you want a reliable local validation signal before
> spending Kaggle submissions.** Each fold trains a full LoRA. Budget:
> ~3-4 hours per fold on A100. With `--folds 3` that's a 9-12h run; on
> Colab Pro free credits this WILL exhaust your session.

The harness:
1. Stratified-splits real `train.csv` by `puzzle_type` into K folds.
2. For each fold: writes a per-fold `train_sft.jsonl` (master jsonl
   minus held-out IDs), trains a LoRA, evaluates on held-out fold via
   vLLM, parses accuracy.
3. Aggregates fold accuracies into `kfold_runs/cv_summary.json`.

The result tells you whether your training recipe is improving for the
right reasons — much more reliable than a single Kaggle submission.

In [ ]:
# Set RUN_KFOLD = True to actually run it. Leave False during normal flow.
RUN_KFOLD = False
KFOLD_N = 3
KFOLD_SKIP_EVAL = False  # set True if vLLM is unavailable in this env

if RUN_KFOLD:
    import subprocess, sys
    cmd = [
        sys.executable, "scripts/06_kfold_cv.py",
        "--folds", str(KFOLD_N),
        "--seed", "42",
        "--data-dir", "data",
        "--train-sft", "data/train_sft.jsonl",
        "--work-dir", "kfold_runs",
        "--model-path", str(MODEL_PATH_LOCAL),
        "--lora-target-mode", LORA_TARGET_MODE,
        "--lora-alpha", str(LORA_ALPHA),
        "--max-seq-length", str(TRAIN_MAX_SEQ),
        "--batch-size", str(TRAIN_BATCH),
        "--grad-accum", str(GRAD_ACCUM),
        "--epochs", str(NUM_EPOCHS),
        "--lr", str(LR),
    ]
    if KFOLD_SKIP_EVAL:
        cmd.append("--skip-eval")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("RUN_KFOLD is False — skipping. Set RUN_KFOLD=True to opt in.")


## (Optional) Phase 7 — Multi-seed LoRA training + weight averaging

> **Skip unless you have credits to spare.** This trains the SAME data
> N times with different seeds and averages the resulting LoRA matrices
> into a single submission adapter ('Model Soups' style). Expected
> impact: +0.02 to +0.05 over a single-seed run.

Cost is N × the single-seed training time. With NUM_EPOCHS=2 and
SYNTHETIC_PER_KIND=400 plus pseudo-labels, expect ~3-4h per seed on
A100. NUM_SEEDS=3 fits inside one Colab Pro session if started early.

After this cell finishes, `lora_adapter/` contains the averaged
adapter, and Phase 4 (package submission) will pick it up
automatically.

In [ ]:
# Set RUN_MULTI_SEED = True to actually run it.
RUN_MULTI_SEED = False
NUM_SEEDS = 3
BASE_SEED = 42

if RUN_MULTI_SEED:
    import subprocess, sys
    cmd = [
        sys.executable, "scripts/07_multi_seed_average.py",
        "--num-seeds", str(NUM_SEEDS),
        "--base-seed", str(BASE_SEED),
        "--data-path", "data/train_sft.jsonl",
        "--intermediate-dir", "multi_seed_runs",
        "--output-dir", "lora_adapter",
        "--model-path", str(MODEL_PATH_LOCAL),
        "--lora-target-mode", LORA_TARGET_MODE,
        "--lora-alpha", str(LORA_ALPHA),
        "--max-seq-length", str(TRAIN_MAX_SEQ),
        "--batch-size", str(TRAIN_BATCH),
        "--grad-accum", str(GRAD_ACCUM),
        "--epochs", str(NUM_EPOCHS),
        "--lr", str(LR),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Averaged adapter is now at lora_adapter/ — Phase 4 will package it.")
else:
    print("RUN_MULTI_SEED is False — skipping. Set RUN_MULTI_SEED=True to opt in.")
